In [43]:
# ============================================================
# MONTAR DRIVE + VERIFICAR GPU (ejecutar primero)
# ============================================================
import torch
from google.colab import drive

# Montar Drive
drive.mount('/content/drive', force_remount=False)

# Verificar GPU
device = 'CUDA' if torch.cuda.is_available() else 'CPU'
print(f"⚡ Dispositivo: {device}")
if device == 'CPU':
    raise RuntimeError("❌ GPU no disponible — cambia runtime a T4 GPU")
else:
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

from pathlib import Path
BASE = Path("/content/drive/MyDrive/TFM/EXPRESIONES")
print(f"\nBASE: {BASE}")
print(f"  Existe: {BASE.exists()}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
⚡ Dispositivo: CUDA
  GPU: Tesla T4
  VRAM: 15.6 GB

BASE: /content/drive/MyDrive/TFM/EXPRESIONES
  Existe: True


# 🔗 Split de Dataset: YOLO_EXPRESIONES_UNIFIED

**TFM — Sistema de Alertas por Expresiones Faciales**

**Estructura real verificada en Google Drive:**
```
My Drive/TFM/EXPRESIONES/
└── YOLO_EXPRESIONES_UNIFIED/
├── images/
│   ├── train/          ← 70% imágenes de entrenamiento  (16,824)
│   ├── val/            ← 15% imágenes de validación      (3,606)
│   └── test/           ← 15% imágenes de prueba          (3,605)
└── labels/
├── train/          ← 70% labels YOLO .txt           (16,824)
├── val/            ← 15% labels YOLO .txt            (3,606)
└── test/           ← 15% labels YOLO .txt            (3,605)
```
**Objetivo:** Tomar los 24,035 pares imagen+label de `YOLO_EXPRESIONES_UNIFIED/images/train` y `labels/train`, y redistribuirlos en tres splits: entrenamiento (**70%**), validación (**15%**) y prueba (**15%**), con paridad garantizada imagen/label en cada split.

> ⚡ **Entorno:** GPU T4 — *Runtime → Change runtime type → T4 GPU*

In [215]:
import os
path = "/content/drive/MyDrive/TFM/EXPRESIONES/YOLO_EXPRESIONES_UNIFIED/images/train"
count = len([f for f in os.listdir(path) if not f.startswith('.')])
print(f"Imágenes en train: {count:,}")

Imágenes en train: 12,492


In [216]:
import os
path = "/content/drive/MyDrive/TFM/EXPRESIONES/YOLO_EXPRESIONES_UNIFIED/labels/train"
count = len([f for f in os.listdir(path) if not f.startswith('.')])
print(f"labels en train: {count:,}")

labels en train: 12,905


In [217]:
# ============================================================
# DIAGNÓSTICO RÁPIDO — estado actual de YOLO_DATASET_UNIFIED
# Solo usa os.listdir (sin conteo de archivos grandes)
# ============================================================
import os
from pathlib import Path

BASE = Path("/content/drive/MyDrive/TFM/EXPRESIONES")
YOLO = BASE / "YOLO_EXPRESIONES_UNIFIED"
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def quick_count(p):
    """Cuenta archivos en UN nivel, sin recursión."""
    try:
        entries = os.listdir(p)
        imgs = sum(1 for e in entries if Path(e).suffix.lower() in IMG_EXTS and not e.startswith("._"))
        txts = sum(1 for e in entries if e.endswith(".txt") and not e.startswith("._"))
        dirs = [e for e in entries if (p / e).is_dir()]
        return imgs, txts, dirs
    except:
        return 0, 0, []

print("=" * 60)
print("YOLO_EXPRESIONES_UNIFIED/images/")
imgs_root, _, subdirs_img = quick_count(YOLO / "images")
print(f"  Archivos en RAÍZ    : {imgs_root:,} imágenes")
print(f"  Subcarpetas         : {subdirs_img}")
for sd in subdirs_img:
    n, _, _ = quick_count(YOLO / "images" / sd)
    print(f"    └── {sd}/  {n:,} imágenes")

print()
print("YOLO_EXPRESIONES_UNIFIED/labels/")
_, lbls_root, subdirs_lbl = quick_count(YOLO / "labels")
print(f"  Archivos en RAÍZ    : {lbls_root:,} labels .txt")
print(f"  Subcarpetas         : {subdirs_lbl}")
for sd in subdirs_lbl:
    _, n, _ = quick_count(YOLO / "labels" / sd)
    print(f"    └── {sd}/  {n:,} labels")

print()
print("=" * 60)
print("CONCLUSIÓN:")
total_root = imgs_root + lbls_root
if imgs_root == 0 and lbls_root == 0:
    print("✅ No hay archivos en raíz. La estructura ya está correcta.")
    print("   → Ejecuta directamente Celda 6 y Celda 7.")
elif subdirs_img or subdirs_lbl:
    print(f"⚠️  HAY {imgs_root:,} imágenes y {lbls_root:,} labels en la raíz")
    print(f"   Y también existen subcarpetas {subdirs_img}.")
    print("   → Necesitas la Celda 4-A para reorganizar.")
else:
    print(f"ℹ️  Solo hay raíz (sin subcarpetas) — estructura plana.")
    print("   → Ejecuta directamente Celda 6 y Celda 7.")

YOLO_EXPRESIONES_UNIFIED/images/
  Archivos en RAÍZ    : 0 imágenes
  Subcarpetas         : ['train', 'val', 'test']
    └── train/  12,492 imágenes
    └── val/  3,040 imágenes
    └── test/  3,366 imágenes

YOLO_EXPRESIONES_UNIFIED/labels/
  Archivos en RAÍZ    : 0 labels .txt
  Subcarpetas         : ['train', 'val', 'test']
    └── train/  12,905 labels
    └── val/  3,040 labels
    └── test/  3,366 labels

CONCLUSIÓN:
✅ No hay archivos en raíz. La estructura ya está correcta.
   → Ejecuta directamente Celda 6 y Celda 7.


In [218]:
# ============================================================
# Conteo de archivos en carpetas específicas
# ============================================================
import time
from pathlib import Path

BASE = Path("/content/drive/MyDrive/TFM/EXPRESIONES")
YOLO = BASE / "YOLO_EXPRESIONES_UNIFIED"
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def count_files_in_dir(directory_path, file_type="any"):
    """Cuenta archivos en un directorio, opcionalmente filtrando por tipo."""
    if not directory_path.exists():
        return 0, f"Carpeta no encontrada: {directory_path}"

    try:
        count = 0
        for entry in directory_path.iterdir():  # iterdir() no usa patrones — maneja espacios en nombres
            if not entry.is_file():
                continue
            if file_type == "image":
                if entry.suffix.lower() in IMG_EXTS:
                    count += 1
            elif file_type == "label":
                if entry.suffix.lower() == ".txt":
                    count += 1
            else:
                count += 1
        return count, None

    except OSError:
        time.sleep(2)
        try:
            count = 0
            for entry in directory_path.iterdir():
                if not entry.is_file():
                    continue
                if file_type == "image":
                    if entry.suffix.lower() in IMG_EXTS:
                        count += 1
                elif file_type == "label":
                    if entry.suffix.lower() == ".txt":
                        count += 1
                else:
                    count += 1
            return count, None
        except OSError:
            return 0, f"Error de I/O (Drive desconectado?): {directory_path}"

print("📊 Conteo de archivos en directorios:")
print("=" * 40)

paths = [
    (YOLO / "images" / "train", "image", "imágenes"),
    (YOLO / "images" / "val",   "image", "imágenes"),
    (YOLO / "images" / "test",   "image", "imágenes"),
    (YOLO / "labels" / "train", "label", "labels (.txt)"),
    (YOLO / "labels" / "val",   "label", "labels (.txt)"),
    (YOLO / "labels" / "test",   "label", "labels (.txt)"),
]

for path, ftype, label in paths:
    count, error = count_files_in_dir(path, file_type=ftype)
    if error:
        print(f"  ❌ {error}")
    else:
        print(f"  📁 {path.relative_to(BASE)}: {count:,} {label}")

print("=" * 40)

📊 Conteo de archivos en directorios:
  📁 YOLO_EXPRESIONES_UNIFIED/images/train: 21,544 imágenes
  📁 YOLO_EXPRESIONES_UNIFIED/images/val: 4,852 imágenes
  📁 YOLO_EXPRESIONES_UNIFIED/images/test: 5,133 imágenes
  📁 YOLO_EXPRESIONES_UNIFIED/labels/train: 21,346 labels (.txt)
  📁 YOLO_EXPRESIONES_UNIFIED/labels/val: 4,852 labels (.txt)
  📁 YOLO_EXPRESIONES_UNIFIED/labels/test: 5,133 labels (.txt)


In [220]:
# ============================================================
# Split dataset: 70% train / 15% val / 15% test
# VERSIÓN SEGURA — verifica antes de borrar
# ============================================================
import shutil
import random
from pathlib import Path

BASE  = Path("/content/drive/MyDrive/TFM/EXPRESIONES")
YOLO  = BASE / "YOLO_EXPRESIONES_UNIFIED"

SRC_IMAGES = YOLO / "images" / "train"
SRC_LABELS = YOLO / "labels" / "train"

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
SPLIT_RATIOS = {"train": 0.70, "val": 0.15, "test": 0.15}
SEED = 42

# ── 1. Pares válidos ──────────────────────────────────────────
all_stems = sorted(
    p.stem for p in SRC_IMAGES.iterdir()
    if p.is_file() and p.suffix.lower() in IMG_EXTS
    and (SRC_LABELS / (p.stem + ".txt")).exists()
)
print(f"Pares imagen+label encontrados: {len(all_stems):,}")

# ── 2. Shuffle y split ────────────────────────────────────────
random.seed(SEED)
random.shuffle(all_stems)

n_total = len(all_stems)
n_train = int(n_total * SPLIT_RATIOS["train"])
n_val   = int(n_total * SPLIT_RATIOS["val"])

splits = {
    "train": all_stems[:n_train],
    "val":   all_stems[n_train : n_train + n_val],
    "test":  all_stems[n_train + n_val:],
}

for split, stems in splits.items():
    print(f"  {split}: {len(stems):,} muestras")

# ── 3. Crear carpetas destino ─────────────────────────────────
for split in splits:
    (YOLO / "images" / split).mkdir(parents=True, exist_ok=True)
    (YOLO / "labels" / split).mkdir(parents=True, exist_ok=True)

# ── 4. Copiar val y test CON verificación ─────────────────────
def find_image(src_dir: Path, stem: str) -> Path | None:
    for ext in IMG_EXTS:
        p = src_dir / (stem + ext)
        if p.exists():
            return p
    return None

copy_errors = []
copy_ok = {"val": [], "test": []}

for split in ("val", "test"):
    dst_img = YOLO / "images" / split
    dst_lbl = YOLO / "labels" / split

    for stem in splits[split]:
        img_src = find_image(SRC_IMAGES, stem)
        lbl_src = SRC_LABELS / (stem + ".txt")

        if img_src is None or not lbl_src.exists():
            copy_errors.append(stem)
            continue

        img_dst = dst_img / img_src.name
        lbl_dst = dst_lbl / lbl_src.name

        shutil.copy2(img_src, img_dst)
        shutil.copy2(lbl_src, lbl_dst)

        # ── Verificar que llegaron y tienen tamaño > 0 ────────
        if img_dst.exists() and lbl_dst.exists() \
           and img_dst.stat().st_size > 0 \
           and lbl_dst.stat().st_size > 0:
            copy_ok[split].append(stem)
        else:
            copy_errors.append(stem)

print(f"\nCopias verificadas — val: {len(copy_ok['val']):,} | test: {len(copy_ok['test']):,}")

# ── 5. Borrar de train SOLO si la copia fue verificada ────────
if copy_errors:
    print(f"\n🚨 {len(copy_errors)} stems fallaron la copia. NO se borra nada de train.")
    print("   Revisa los errores antes de continuar:")
    print(f"   {copy_errors[:10]}")
else:
    print("\nTodas las copias verificadas. Procediendo a limpiar train...")

    verified = set(copy_ok["val"]) | set(copy_ok["test"])
    deleted = 0

    for p in list(SRC_IMAGES.iterdir()):
        if p.is_file() and p.suffix.lower() in IMG_EXTS and p.stem in verified:
            p.unlink()
            deleted += 1

    for p in list(SRC_LABELS.iterdir()):
        if p.is_file() and p.suffix.lower() == ".txt" and p.stem in verified:
            p.unlink()
            deleted += 1

    print(f"  Archivos eliminados de train: {deleted:,}")

# ── 6. Resumen final ──────────────────────────────────────────
print("\n📊 Verificación final:")
print("=" * 45)
for split in ("train", "val", "test"):
    n_img = sum(1 for p in (YOLO / "images" / split).iterdir()
                if p.is_file() and p.suffix.lower() in IMG_EXTS)
    n_lbl = sum(1 for p in (YOLO / "labels" / split).iterdir()
                if p.is_file() and p.suffix.lower() == ".txt")
    status = "✅" if n_img == n_lbl else "⚠️  DESBALANCE"
    print(f"  {status} {split:5s} → {n_img:,} imágenes | {n_lbl:,} labels")

if copy_errors:
    print(f"\n⚠️  {len(copy_errors)} stems con error de copia.")
else:
    print("\n✅ Split completado sin errores.")
print("=" * 45)

Pares imagen+label encontrados: 18,518
  train: 12,962 muestras
  val: 2,777 muestras
  test: 2,779 muestras

Copias verificadas — val: 2,777 | test: 2,779

Todas las copias verificadas. Procediendo a limpiar train...
  Archivos eliminados de train: 11,112

📊 Verificación final:
  ⚠️  DESBALANCE train → 15,988 imágenes | 15,790 labels
  ✅ val   → 7,295 imágenes | 7,295 labels
  ✅ test  → 7,604 imágenes | 7,604 labels

✅ Split completado sin errores.


In [221]:
from pathlib import Path
from google.colab import drive

# 1. Montar Drive si no está montado
drive.mount('/content/drive', force_remount=False)

# 2. Verificar ruta raíz
root = Path("/content/drive/MyDrive")
print("📁 Carpetas en MyDrive:")
for f in sorted(root.iterdir()):
    if f.is_dir():
        print(f"  {repr(f.name)}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📁 Carpetas en MyDrive:
  'Colab Notebooks'
  'TFM'


In [222]:
from pathlib import Path

tfm = Path("/content/drive/MyDrive/TFM")
print("📁 Carpetas en TFM:")
for f in sorted(tfm.iterdir()):
    if f.is_dir():
        print(f"  {repr(f.name)}")

📁 Carpetas en TFM:
  'CAIDAS'
  'EXPRESIONES'


In [223]:
from pathlib import Path

exp = Path("/content/drive/MyDrive/TFM/EXPRESIONES")
print("📁 Carpetas en EXPRESIONES:")
for f in sorted(exp.iterdir()):
    if f.is_dir():
        print(f"  {repr(f.name)}")

📁 Carpetas en EXPRESIONES:
  'YOLO_EXPRESIONES_UNIFIED'


In [224]:
# ============================================================
# CORRECCIÓN Y REDISTRIBUCIÓN COMPLETA 70/15/15
# Los archivos descartados van a _descartados/ como respaldo
# ============================================================
import shutil
import random
from pathlib import Path
from datetime import datetime

BASE     = Path("/content/drive/MyDrive/TFM/EXPRESIONES")
YOLO     = BASE / "YOLO_EXPRESIONES_UNIFIED"
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
SEED     = 42
SPLIT_RATIOS = {"train": 0.70, "val": 0.15, "test": 0.15}

# Carpeta de respaldo con timestamp para no pisar respaldos anteriores
TIMESTAMP  = datetime.now().strftime("%Y%m%d_%H%M%S")
BACKUP_DIR = YOLO / f"_descartados_{TIMESTAMP}"
BACKUP_IMG = BACKUP_DIR / "images"
BACKUP_LBL = BACKUP_DIR / "labels"
BACKUP_IMG.mkdir(parents=True, exist_ok=True)
BACKUP_LBL.mkdir(parents=True, exist_ok=True)

print("=" * 55)
print("  REDISTRIBUCIÓN 70/15/15 CON RESPALDO DE DESCARTADOS")
print("=" * 55)
print(f"  Carpeta de respaldo: {BACKUP_DIR.name}")

# ── 1. Recopilar TODOS los pares válidos de las 3 carpetas ───
print("\n[1/5] Recolectando pares válidos...")

all_pairs = {}  # stem -> (img_path, lbl_path)

for split in ("train", "val", "test"):
    img_dir = YOLO / "images" / split
    lbl_dir = YOLO / "labels" / split
    if not img_dir.exists():
        continue
    found = 0
    for p in img_dir.iterdir():
        if not p.is_file() or p.suffix.lower() not in IMG_EXTS:
            continue
        lbl = lbl_dir / (p.stem + ".txt")
        if lbl.exists() and p.stem not in all_pairs:
            all_pairs[p.stem] = (p, lbl)
            found += 1
    print(f"  {split:5s}: {found:,} pares encontrados")

print(f"\n  Total pares únicos: {len(all_pairs):,}")

# ── 2. Calcular distribución esperada ─────────────────────────
n_total = len(all_pairs)
n_train = int(n_total * SPLIT_RATIOS["train"])
n_val   = int(n_total * SPLIT_RATIOS["val"])
n_test  = n_total - n_train - n_val

print(f"\n[2/5] Distribución esperada (70/15/15):")
print(f"  train : {n_train:,}  ({n_train/n_total*100:.1f}%)")
print(f"  val   : {n_val:,}  ({n_val/n_total*100:.1f}%)")
print(f"  test  : {n_test:,}  ({n_test/n_total*100:.1f}%)")

# ── 3. Shuffle con seed fijo ──────────────────────────────────
stems = sorted(all_pairs.keys())
random.seed(SEED)
random.shuffle(stems)

new_splits = {
    "train": set(stems[:n_train]),
    "val":   set(stems[n_train : n_train + n_val]),
    "test":  set(stems[n_train + n_val:]),
}

# Mapa inverso: stem -> split asignado
stem_to_split = {}
for split, split_stems in new_splits.items():
    for stem in split_stems:
        stem_to_split[stem] = split

# ── 4. Mover archivos a su carpeta correcta ───────────────────
print(f"\n[3/5] Redistribuyendo archivos...")

errors  = []
moved   = {"train": 0, "val": 0, "test": 0}
backed  = 0

for stem, (img_src, lbl_src) in all_pairs.items():
    target_split = stem_to_split[stem]
    dst_img = YOLO / "images" / target_split / img_src.name
    dst_lbl = YOLO / "labels" / target_split / lbl_src.name

    try:
        if img_src.resolve() != dst_img.resolve():
            shutil.move(str(img_src), dst_img)
        if lbl_src.resolve() != dst_lbl.resolve():
            shutil.move(str(lbl_src), dst_lbl)
        moved[target_split] += 1
    except Exception as e:
        errors.append((stem, str(e)))

for split, count in moved.items():
    print(f"  {split:5s}: {count:,} archivos en su lugar")

# ── 5. Detectar y respaldar huérfanos ────────────────────────
print(f"\n[4/5] Detectando huérfanos y enviando a respaldo...")

for split in ("train", "val", "test"):
    img_dir = YOLO / "images" / split
    lbl_dir = YOLO / "labels" / split

    # Imagen sin label
    for p in list(img_dir.iterdir()):
        if not p.is_file() or p.suffix.lower() not in IMG_EXTS:
            continue
        if not (lbl_dir / (p.stem + ".txt")).exists():
            shutil.move(str(p), BACKUP_IMG / p.name)
            backed += 1

    # Label sin imagen
    for p in list(lbl_dir.iterdir()):
        if not p.is_file() or p.suffix.lower() != ".txt":
            continue
        if not any((img_dir / (p.stem + ext)).exists() for ext in IMG_EXTS):
            shutil.move(str(p), BACKUP_LBL / p.name)
            backed += 1

print(f"  {backed:,} archivos huérfanos enviados a respaldo")

# ── 6. Verificación final con porcentajes ─────────────────────
print(f"\n[5/5] Verificación final:")
print("=" * 55)

total_imgs = 0
counts = {}
for split in ("train", "val", "test"):
    n_img = sum(1 for p in (YOLO / "images" / split).iterdir()
                if p.is_file() and p.suffix.lower() in IMG_EXTS)
    n_lbl = sum(1 for p in (YOLO / "labels" / split).iterdir()
                if p.is_file() and p.suffix.lower() == ".txt")
    counts[split] = (n_img, n_lbl)
    total_imgs += n_img

for split in ("train", "val", "test"):
    n_img, n_lbl = counts[split]
    pct    = n_img / total_imgs * 100 if total_imgs > 0 else 0
    status = "✅" if n_img == n_lbl else "⚠️  DESBALANCE"
    print(f"  {status} {split:5s} → {n_img:,} imgs | {n_lbl:,} labels | {pct:.1f}%")

n_bkp_img = sum(1 for p in BACKUP_IMG.iterdir() if p.is_file())
n_bkp_lbl = sum(1 for p in BACKUP_LBL.iterdir() if p.is_file())

print(f"\n  📦 Respaldo en {BACKUP_DIR.name}:")
print(f"     imágenes : {n_bkp_img:,}")
print(f"     labels   : {n_bkp_lbl:,}")
print(f"\n  Total imágenes activas: {total_imgs:,}")

if errors:
    print(f"\n⚠️  {len(errors)} errores durante el movimiento:")
    for stem, err in errors[:5]:
        print(f"    {stem}: {err}")
else:
    print("\n✅ Redistribución completada sin errores.")
print("=" * 55)

  REDISTRIBUCIÓN 70/15/15 CON RESPALDO DE DESCARTADOS
  Carpeta de respaldo: _descartados_20260608_073455

[1/5] Recolectando pares válidos...
  train: 12,962 pares encontrados
  val  : 5,890 pares encontrados
  test : 5,364 pares encontrados

  Total pares únicos: 24,216

[2/5] Distribución esperada (70/15/15):
  train : 16,951  (70.0%)
  val   : 3,632  (15.0%)
  test  : 3,633  (15.0%)

[3/5] Redistribuyendo archivos...
  train: 16,951 archivos en su lugar
  val  : 3,632 archivos en su lugar
  test : 3,633 archivos en su lugar

[4/5] Detectando huérfanos y enviando a respaldo...
  5,493 archivos huérfanos enviados a respaldo

[5/5] Verificación final:
  ✅ train → 16,951 imgs | 16,951 labels | 62.8%
  ✅ val   → 4,685 imgs | 4,685 labels | 17.4%
  ✅ test  → 5,353 imgs | 5,353 labels | 19.8%

  📦 Respaldo en _descartados_20260608_073455:
     imágenes : 2,688
     labels   : 2,805

  Total imágenes activas: 26,989

✅ Redistribución completada sin errores.


In [225]:
# ============================================================
# Conteo de archivos en carpetas específicas
# ============================================================
import time
from pathlib import Path

BASE = Path("/content/drive/MyDrive/TFM/EXPRESIONES")
YOLO = BASE / "YOLO_EXPRESIONES_UNIFIED"
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def count_files_in_dir(directory_path, file_type="any"):
    """Cuenta archivos en un directorio, opcionalmente filtrando por tipo."""
    if not directory_path.exists():
        return 0, f"Carpeta no encontrada: {directory_path}"

    try:
        count = 0
        for entry in directory_path.iterdir():  # iterdir() no usa patrones — maneja espacios en nombres
            if not entry.is_file():
                continue
            if file_type == "image":
                if entry.suffix.lower() in IMG_EXTS:
                    count += 1
            elif file_type == "label":
                if entry.suffix.lower() == ".txt":
                    count += 1
            else:
                count += 1
        return count, None

    except OSError:
        time.sleep(2)
        try:
            count = 0
            for entry in directory_path.iterdir():
                if not entry.is_file():
                    continue
                if file_type == "image":
                    if entry.suffix.lower() in IMG_EXTS:
                        count += 1
                elif file_type == "label":
                    if entry.suffix.lower() == ".txt":
                        count += 1
                else:
                    count += 1
            return count, None
        except OSError:
            return 0, f"Error de I/O (Drive desconectado?): {directory_path}"

print("📊 Conteo de archivos en directorios:")
print("=" * 40)

paths = [
    (YOLO / "images" / "train", "image", "imágenes"),
    (YOLO / "images" / "val",   "image", "imágenes"),
    (YOLO / "images" / "test",   "image", "imágenes"),
    (YOLO / "labels" / "train", "label", "labels (.txt)"),
    (YOLO / "labels" / "val",   "label", "labels (.txt)"),
    (YOLO / "labels" / "test",   "label", "labels (.txt)"),
]

for path, ftype, label in paths:
    count, error = count_files_in_dir(path, file_type=ftype)
    if error:
        print(f"  ❌ {error}")
    else:
        print(f"  📁 {path.relative_to(BASE)}: {count:,} {label}")

print("=" * 40)

📊 Conteo de archivos en directorios:
  📁 YOLO_EXPRESIONES_UNIFIED/images/train: 16,951 imágenes
  📁 YOLO_EXPRESIONES_UNIFIED/images/val: 4,685 imágenes
  📁 YOLO_EXPRESIONES_UNIFIED/images/test: 5,353 imágenes
  📁 YOLO_EXPRESIONES_UNIFIED/labels/train: 16,951 labels (.txt)
  📁 YOLO_EXPRESIONES_UNIFIED/labels/val: 4,685 labels (.txt)
  📁 YOLO_EXPRESIONES_UNIFIED/labels/test: 5,353 labels (.txt)
